[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 中等：多头交叉注意力

实现 **多头交叉注意力**（编码器-解码器注意力）。

### 函数签名
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — 解码器查询
        # x_kv: (B, S_kv, D) — 编码器键/值
```

### 与自注意力的关键区别
- Q 来自解码器，K 和 V 来自编码器
- 无因果掩码（所有编码器位置可见）

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✏️ 在此实现你的代码

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        pass  # W_q, W_k, W_v, W_o

    def forward(self, x_q, x_kv):
        pass  # Q 来自 x_q，K/V 来自 x_kv，无因果掩码

- **B**: 批次大小
- **S_q**: 查询序列长度
- **S_kv**: 键/值序列长度
- **D**: 模型特征维度
- **H**: 注意力头数
- **d_k**: 每个头的特征维度
- **Q**: 查询矩阵
- **K**: 键矩阵
- **V**: 值矩阵
- **scores**: 注意力分数
- **attn_weights**: 注意力权重
- **attn_output**: 注意力输出
- **output**: 最终输出
- **x_q**: 输入查询（来自解码器）
- **x_kv**: 输入键/值（来自编码器）
- **W_q**: 查询投影权重(权重张量)
- **W_k**: 键投影权重(权重张量)
- **W_v**: 值投影权重(权重张量)
- **W_o**: 输出投影权重(权重张量)

In [ ]:
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        """
        多头交叉注意力（编码器-解码器注意力）
        
        Args:
            d_model: 模型维度
            num_heads: 注意力头数
        """
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x_q: torch.Tensor, x_kv: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x_q: (B, S_q, D) — 解码器查询
            x_kv: (B, S_kv, D) — 编码器键/值
        
        Returns:
            Tensor: (B, S_q, D) — 交叉注意力输出
        """
        B, S_q, D = x_q.shape
        B, S_kv, _ = x_kv.shape
        
        # 1. 线性投影并重塑为多头形式
        # Q: (B, S_q, D) -> (B, S_q, num_heads, d_k) -> (B, num_heads, S_q, d_k)
        Q = self.W_q(x_q).view(B, S_q, self.num_heads, self.d_k).transpose(1, 2)
        
        # K, V: (B, S_kv, D) -> (B, S_kv, num_heads, d_k) -> (B, num_heads, S_kv, d_k)
        K = self.W_k(x_kv).view(B, S_kv, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x_kv).view(B, S_kv, self.num_heads, self.d_k).transpose(1, 2)
        
        # 2. 计算缩放点积注意力
        # Q @ K^T: (B, num_heads, S_q, d_k) @ (B, num_heads, d_k, S_kv) -> (B, num_heads, S_q, S_kv)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        '''
            # 创建因果掩码矩阵
            mask = torch.triu(torch.ones(S_q, S_kv), diagonal=1).bool()
            # 对于 S_q=3, S_kv=5 的掩码示例：
            # [[0, 0, 0, 0, 0],
            #  [1, 0, 0, 0, 0],
            #  [1, 1, 0, 0, 0]]
            
            # 将掩码位置设为 -inf
            scores = scores.masked_fill(mask, float('-inf'))
        '''
        
        # 3. 应用 softmax（无因果掩码，所有编码器位置可见）
        # 手动实现 softmax 而不使用 F.softmax
        exp_scores = torch.exp(scores - scores.max(dim=-1, keepdim=True)[0])  # 数值稳定
        attn_weights = exp_scores / exp_scores.sum(dim=-1, keepdim=True)  # (B, num_heads, S_q, S_kv)
        
        # 4. 加权聚合值
        # (B, num_heads, S_q, S_kv) @ (B, num_heads, S_kv, d_k) 
        # -> (B, num_heads, S_q, d_k)
        attn_output = torch.matmul(attn_weights, V)
        
        # 5. 合并多头
        # (B, num_heads, S_q, d_k) -> (B, S_q, num_heads, d_k) -> (B, S_q, D)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, S_q, D)
        
        # 6. 最终线性投影
        output = self.W_o(attn_output)
        
        return output

In [ ]:
# 🧪 调试
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('输出:', attn(x_q, x_kv).shape)

In [ ]:
# ✅ 提交
from torch_judge import check
check('cross_attention')